# PS-3 follow-up — beam search vs greedy decoding

Your trained Seq2Seq produced output like this:

> `we're sorry sorry for the problems you weren't able to use the to the your to the the to the you you the the the`

That is not a training failure — it is a **decoding** failure. Greedy decoding picks the single
highest-probability word at every step and never reconsiders. Once an undertrained decoder drifts
into a high-probability rut ("the" → "the" → "the"), greedy has no way out.

**Beam search** keeps several candidate sentences alive at once and scores each *whole sequence*,
so a word that looks best right now can lose to one that leads somewhere better. Two additions
here matter as much as the beam itself:

- **n-gram blocking** — a beam may not repeat a 3-gram it has already produced. This kills the
  `the the the` loop outright.
- **Length normalisation** — beam search otherwise prefers short sequences, because every extra
  word adds a negative log-probability. Dividing by `length^0.7` corrects that bias.

**No retraining.** This reloads the checkpoint your last run saved to Drive and only re-decodes,
so it takes a couple of minutes. You get a greedy-vs-beam comparison table, which is exactly the
kind of before/after evidence that strengthens a report.

**Run the cells in order.** Set the runtime to a T4 GPU first (Runtime → Change runtime type).

In [ ]:
!pip install -q rouge-score
import nltk; nltk.download('punkt', quiet=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/support_ticket_project'
CKPT  = f'{DRIVE}/checkpoints/ps3_seq2seq_best.pt'

import os
assert os.path.exists(CKPT), f'Checkpoint not found at {CKPT} - run notebook 05 first.'
print('checkpoint found')

In [ ]:
import random, time
from collections import Counter

import numpy as np
import pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PAD, UNK, SOS, EOS = 0, 1, 2, 3

# These MUST match the values used in notebook 05, or the saved weights will not
# load into the model you build below.
EMB_S2S, HID_S2S = 256, 512
MAXLEN_SRC, MAXLEN_TGT = 40, 40

# Beam search runs one sentence at a time, so it is slower than greedy. 1000 test
# rows is plenty to compare the two fairly; raise it if you have time to spare.
N_EVAL = 1000
BEAM_SIZE = 3
print(DEV)

### Model definition — identical to notebook 05, so the checkpoint loads

In [ ]:
def tokenize(t):
    return str(t).lower().split()

class Vocab:
    def __init__(self, texts=None, max_size=20000, min_freq=2, itos=None):
        if itos is not None:                      # rebuild from a checkpoint
            self.itos = list(itos)
        else:
            counts = Counter(w for t in texts for w in tokenize(t))
            keep = [w for w, n in counts.most_common() if n >= min_freq][: max_size - 4]
            self.itos = ['<pad>', '<unk>', '<sos>', '<eos>'] + keep
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, maxlen, sos=False, eos=False):
        ids = [self.stoi.get(w, UNK) for w in tokenize(text)][: maxlen - int(sos) - int(eos)]
        if sos: ids = [SOS] + ids
        if eos: ids = ids + [EOS]
        return ids + [PAD] * (maxlen - len(ids)), len(ids)

    def decode(self, ids):
        return ' '.join(self.itos[i] for i in ids if i > EOS)


class Encoder(nn.Module):
    def __init__(self, V, emb, hid):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.rnn = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.fh = nn.Linear(hid * 2, hid)
        self.fcell = nn.Linear(hid * 2, hid)

    def forward(self, x, lengths):
        e = self.emb(x)
        p = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(p)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        h = torch.tanh(self.fh(torch.cat([h[-2], h[-1]], 1))).unsqueeze(0)
        c = torch.tanh(self.fcell(torch.cat([c[-2], c[-1]], 1))).unsqueeze(0)
        return out, (h, c)


class Attention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.W = nn.Linear(hid * 2 + hid, hid)
        self.v = nn.Linear(hid, 1, bias=False)

    def forward(self, dec_h, enc_out, mask):
        T = enc_out.size(1)
        d = dec_h.repeat(T, 1, 1).transpose(0, 1)
        e = self.v(torch.tanh(self.W(torch.cat([d, enc_out], 2)))).squeeze(2)
        return torch.softmax(e.masked_fill(~mask, -1e9), dim=1)


class Decoder(nn.Module):
    def __init__(self, V, emb, hid):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.att = Attention(hid)
        self.rnn = nn.LSTM(emb + hid * 2, hid, batch_first=True)
        self.fc = nn.Linear(hid * 3 + emb, V)

    def forward(self, token, hidden, enc_out, mask):
        e = self.emb(token).unsqueeze(1)
        a = self.att(hidden[0][-1], enc_out, mask).unsqueeze(1)
        ctx = torch.bmm(a, enc_out)
        out, hidden = self.rnn(torch.cat([e, ctx], 2), hidden)
        pred = self.fc(torch.cat([out.squeeze(1), ctx.squeeze(1), e.squeeze(1)], 1))
        return pred, hidden


class Seq2Seq(nn.Module):
    def __init__(self, V, emb=EMB_S2S, hid=HID_S2S):
        super().__init__()
        self.enc = Encoder(V, emb, hid)
        self.dec = Decoder(V, emb, hid)
        self.V = V

    @torch.no_grad()
    def generate(self, src, slen, maxlen=MAXLEN_TGT):
        '''Greedy decoding - the baseline we are trying to beat.'''
        enc_out, hidden = self.enc(src, slen)
        mask = (src != PAD)
        token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
        done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
        res = []
        for _ in range(maxlen):
            pred, hidden = self.dec(token, hidden, enc_out, mask)
            token = pred.argmax(1)
            token = torch.where(done, torch.full_like(token, PAD), token)
            done = done | (token == EOS)
            res.append(token)
            if done.all():
                break
        return torch.stack(res, 1)

In [ ]:
ck = torch.load(CKPT, map_location=DEV)
vocab = Vocab(itos=ck['itos'])
model = Seq2Seq(len(vocab)).to(DEV)
model.load_state_dict(ck['model'])
model.eval()
print(f"loaded checkpoint  ·  vocab {len(vocab):,}  ·  val_loss {ck['val_loss']:.4f}")

### Rebuild the same test split notebook 05 used

In [ ]:
df3 = pd.read_csv(f'{DRIVE}/colab_ps3.csv').dropna()

# Same call, same seed, same test_size as notebook 05 -> the same held-out rows.
# If any of those three differ you would be scoring on data the model trained on.
_, src_te, _, tgt_te = train_test_split(
    df3['customer_text'].values, df3['response_text'].values,
    test_size=0.1, random_state=SEED)

src_te, tgt_te = src_te[:N_EVAL], tgt_te[:N_EVAL]
print(f'{len(src_te):,} test rows for the comparison')

### The beam search itself

At each step every live beam proposes its best continuations; all candidates are scored by the
running total log-probability and only the top `k` survive. A beam that emits `<eos>` is retired
into `finished` with its score divided by `length^0.7`.

The n-gram block is the loop-killer: before choosing, any token that would recreate a 3-gram the
beam has already produced is set to negative infinity.

In [ ]:
@torch.no_grad()
def beam_generate(model, src_row, slen_row, beam_size=BEAM_SIZE, maxlen=MAXLEN_TGT,
                  length_alpha=0.7, no_repeat_ngram=3):
    src = src_row.unsqueeze(0)
    enc_out, hidden = model.enc(src, slen_row.unsqueeze(0))
    mask = (src != PAD)
    k = beam_size

    # every beam needs its own copy of the encoder output and hidden state
    enc_k = enc_out.expand(k, -1, -1).contiguous()
    mask_k = mask.expand(k, -1).contiguous()
    h, c = hidden
    hid_k = (h.expand(-1, k, -1).contiguous(), c.expand(-1, k, -1).contiguous())

    seqs = [[SOS] for _ in range(k)]
    scores = torch.full((k,), -1e9, device=src.device)
    scores[0] = 0.0                      # only beam 0 is live at the first step
    finished = []

    for step in range(maxlen):
        tok = torch.tensor([s[-1] for s in seqs], device=src.device)
        logits, hid_k = model.dec(tok, hid_k, enc_k, mask_k)
        logp = torch.log_softmax(logits, dim=-1)

        if no_repeat_ngram and step + 1 >= no_repeat_ngram:
            for b, s in enumerate(seqs):
                prefix = tuple(s[-(no_repeat_ngram - 1):])
                for i in range(len(s) - no_repeat_ngram + 1):
                    if tuple(s[i:i + no_repeat_ngram - 1]) == prefix:
                        logp[b, s[i + no_repeat_ngram - 1]] = -1e9

        cand = scores.unsqueeze(1) + logp
        top_scores, top_idx = cand.view(-1).topk(k)
        beam_idx = torch.div(top_idx, model.V, rounding_mode='floor')
        tok_idx = top_idx % model.V

        new_seqs, keep_beam, keep_scores = [], [], []
        for s_, b_, t_ in zip(top_scores.tolist(), beam_idx.tolist(), tok_idx.tolist()):
            seq = seqs[b_] + [t_]
            if t_ == EOS:
                finished.append((s_ / (len(seq) ** length_alpha), seq))
            else:
                new_seqs.append(seq); keep_beam.append(b_); keep_scores.append(s_)

        if not new_seqs or len(finished) >= k:
            break
        while len(new_seqs) < k:
            new_seqs.append(new_seqs[-1]); keep_beam.append(keep_beam[-1])
            keep_scores.append(-1e9)

        seqs = new_seqs
        scores = torch.tensor(keep_scores[:k], device=src.device)
        sel = torch.tensor(keep_beam[:k], device=src.device)
        hid_k = (hid_k[0][:, sel, :].contiguous(), hid_k[1][:, sel, :].contiguous())

    if not finished:
        finished = [(scores[i].item() / (len(seqs[i]) ** length_alpha), seqs[i])
                    for i in range(len(seqs))]
    finished.sort(key=lambda x: x[0], reverse=True)
    return finished[0][1][1:]            # drop the leading <sos>

In [ ]:
# encode the test inputs once, reuse for both decoders
enc = [vocab.encode(t, MAXLEN_SRC) for t in src_te]
src_ids = torch.tensor([e[0] for e in enc], device=DEV)
src_len = torch.tensor([max(e[1], 1) for e in enc])

# ---- greedy (batched, fast) ----
t0 = time.time()
greedy_out = []
for i in range(0, len(src_ids), 128):
    g = model.generate(src_ids[i:i + 128], src_len[i:i + 128])
    greedy_out += [vocab.decode(r.tolist()) for r in g.cpu()]
print(f'greedy done in {time.time() - t0:.0f}s')

# ---- beam (one at a time, slower) ----
t0 = time.time()
beam_out = []
for i in range(len(src_ids)):
    beam_out.append(vocab.decode(beam_generate(model, src_ids[i], src_len[i])))
    if (i + 1) % 250 == 0:
        print(f'  beam {i + 1}/{len(src_ids)}  ({time.time() - t0:.0f}s)')
print(f'beam done in {time.time() - t0:.0f}s')

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

sm = SmoothingFunction().method4
rs = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
refs = [str(t) for t in tgt_te]

def repeated_ngram_rate(hyps, n=3):
    '''Share of outputs containing the same n-gram twice - the degeneration symptom.'''
    bad = 0
    for h in hyps:
        w = h.split()
        grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        if grams and len(grams) != len(set(grams)):
            bad += 1
    return bad / max(len(hyps), 1)

def report(name, hyps):
    bleu = corpus_bleu([[r.split()] for r in refs], [h.split() for h in hyps],
                       smoothing_function=sm)
    idx = random.sample(range(len(refs)), min(1000, len(refs)))
    r1 = np.mean([rs.score(refs[i], hyps[i])['rouge1'].fmeasure for i in idx])
    rl = np.mean([rs.score(refs[i], hyps[i])['rougeL'].fmeasure for i in idx])
    return {
        'decoder': name,
        'BLEU-4': round(bleu, 4),
        'ROUGE-1': round(r1, 4),
        'ROUGE-L': round(rl, 4),
        'unique %': round(100 * len(set(hyps)) / len(hyps), 1),
        'avg words': round(np.mean([len(h.split()) for h in hyps]), 1),
        'repeated 3-gram %': round(100 * repeated_ngram_rate(hyps), 1),
    }

comparison = pd.DataFrame([report('greedy', greedy_out),
                           report(f'beam (k={BEAM_SIZE})', beam_out)])
comparison.to_csv(f'{DRIVE}/ps3_decoder_comparison.csv', index=False)
comparison

### Read the outputs, not just the table

The number to watch is **repeated 3-gram %**. If beam search drops it close to zero, you have
directly fixed the `the the the` failure — and you can say so with evidence.

BLEU may go up only slightly, or even down a little. That is normal and worth explaining: beam
search optimises sequence probability, not n-gram overlap with one particular reference.

In [ ]:
for i in range(8):
    print('Q    :', src_te[i][:150])
    print('REF  :', refs[i][:150])
    print('GRD  :', greedy_out[i][:150])
    print('BEAM :', beam_out[i][:150])
    print('-' * 95)

---
## For your report

Put the comparison table in the PS-3 section with two or three of the sample blocks above, and
make these points:

1. **The repetition was a decoding artefact, not a training one.** The same weights produce
   degenerate output under greedy decoding and cleaner output under beam search with n-gram
   blocking. Quote the repeated-3-gram percentage before and after.
2. **BLEU barely moves.** Explain why: beam search maximises the probability of the whole
   sequence, while BLEU measures overlap with one reference among many valid replies. This is
   evidence that BLEU is a weak proxy for reply quality — which is the argument for reporting
   sample outputs alongside it.
3. **What is still unsolved.** Coherence beyond the first clause is a capacity and data problem,
   not a decoding one. Fixes, in order: more pairs and more epochs, pretrained embeddings, or a
   pretrained transformer (T5/BART) instead of an LSTM trained from scratch.

`ps3_decoder_comparison.csv` is saved to Drive alongside your checkpoints.